# Pregunta 1
Implemente un detector de movimiento por diferencia de cuadros y aplíquelo sobre la secuencia
de imágenes (carpetas mora_mov, sephanoides_mov y vigilancia_mov). Muestre ejemplos de
detección de movimiento y explique la forma que adoptan los pixeles de primer plano.

In [1]:
import cv2 as cv
import numpy as np
import os
from matplotlib import pyplot as plt

In [2]:
def detector_mov_diferencia(file_path, threshold=30):
    # Calcula la diferencia entre cuadros consecutivos para detectar movimiento
    images = sorted([img for img in os.listdir(file_path) if img.endswith((".jpg", ".bmp"))])
    movimiento = []
    for i in range(1, len(images)):
        img1 = cv.imread(os.path.join(file_path, images[i-1]), cv.IMREAD_GRAYSCALE)
        img2 = cv.imread(os.path.join(file_path, images[i]), cv.IMREAD_GRAYSCALE)
        #quiero las solo las differencias negativas 
        diff = cv.subtract(img1, img2)
        _, thresh = cv.threshold(diff, threshold, 255, cv.THRESH_BINARY)
        movimiento.append(thresh)
    return movimiento
labmov = 'Lab-Mov'
mora_path = os.path.join(labmov, 'mora_mov')
sephanoides_path = os.path.join(labmov, 'sephanoides_mov')
vigilancia_path = os.path.join(labmov, 'vigilancia_mov')

mora_dif = detector_mov_diferencia(mora_path)
sephanoides_dif = detector_mov_diferencia(sephanoides_path)
vigilancia_dif = detector_mov_diferencia(vigilancia_path)

In [3]:
# Guardar resultados
def save_results(movimiento, folder_name):
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
    for i, img in enumerate(movimiento):
        cv.imwrite(os.path.join(folder_name, f"mov_{i+1:03d}.jpg"), img)
        
results = 'Results_1'

mora_dif_path = os.path.join(results, 'mora_dif')
sephanoides_dif_path = os.path.join(results, 'sephanoides_dif')
vigilancia_dif_path = os.path.join(results, 'vigilancia_dif')

save_results(mora_dif,mora_dif_path) 
save_results(sephanoides_dif,sephanoides_dif_path) 
save_results(vigilancia_dif,vigilancia_dif_path)


# Pregunta 2
Utilizando la secuencia de imágenes de la carpeta “sephanoides_fondo” y “vigilancia_fondo”,
genere un modelo de fondo de dos matrices calculando el promedio de los cuadros (matriz 1) y
la desviación estándar (matriz 2). Elija un umbral apropiado y aplíquelo para detectar
movimiento sobre las secuencias de imágenes respectivas (sephanoides_mov y vigilancia_mov).
Entregue las matrices del modelo de fondo. Muestre ejemplos de detección de movimiento y
explique la forma que adoptan los pixeles de primer plano.


In [ ]:
#utilizando sephanoides_fondo y vigilancia_fondo. 
#genere un modelo de fondo de dos matrices calculando el promedio de los cuadros (matriz 1) y la desviación estándar (matriz 2)

def modelo_fondo(file_path):
    images = sorted([img for img in os.listdir(file_path) if img.lower().endswith((".jpg", ".bmp"))])
    sum_img = None
    sum_sq_img = None
    count = 0
    for img_name in images:
        img = cv.imread(os.path.join(file_path, img_name), cv.IMREAD_GRAYSCALE)
        img = img.astype(np.float32)

        sum_img = np.zeros_like(img, dtype=np.float32)
        sum_sq_img = np.zeros_like(img, dtype=np.float32)

        sum_img += img
        sum_sq_img += img**2
        count += 1

    # Calcular promedio y desviación estándar
    mean_img = (sum_img / count).astype(np.uint8)
    std_img = np.sqrt((sum_sq_img / count) - (mean_img.astype(np.float32))**2).astype(np.uint8)

    return mean_img, std_img



# Detectar movimiento usando el modelo de fondo, umbralizando la diferencia entre el cuadro actual y el modelo de fondo
def detector_mov_fondo(file_path, mean_img, std_img, alpha=2):
    images = sorted([img for img in os.listdir(file_path) if img.endswith((".jpg", ".bmp"))])
    movimiento = []
    for img_name in images:
        img = cv.imread(os.path.join(file_path, img_name), cv.IMREAD_GRAYSCALE)
        # diff = cv.absdiff(img, mean_img)
        # mean_mat = np.full(img.shape, mean_img, dtype=np.uint8)
        # Umbral por píxel: alpha * desviación estándar
        thresh_num = mean_img - alpha * std_img
        diff = (img )
        
        # Comparación por elemento (matricial)
        thresh = np.where(diff <= thresh_num, 255, 0).astype(np.uint8)

        movimiento.append(thresh)
    return movimiento


In [45]:
# Generar modelos de fondo
sephanoides_background_path = os.path.join(labmov, 'sephanoides_fondo')
vigilancia_background_path = os.path.join(labmov, 'vigilancia_fondo')
sephanoides_mean, sephanoides_std = modelo_fondo(sephanoides_background_path)
vigilancia_mean, vigilancia_std = modelo_fondo(vigilancia_background_path) 

# Detectar movimiento usando el modelo de fondo
sephanoides_fondo = detector_mov_fondo(sephanoides_path, sephanoides_mean, sephanoides_std, alpha = 1.5)
vigilancia_fondo = detector_mov_fondo(vigilancia_path, vigilancia_mean, vigilancia_std, alpha = 1.5)

results = 'Results_2'
# Guardar resultados
sephanoides_fondo_path = os.path.join(results, 'sephanoides_fondo')
vigilancia_fondo_path = os.path.join(results, 'vigilancia_fondo')  
save_results(sephanoides_fondo,sephanoides_fondo_path)
save_results(vigilancia_fondo,vigilancia_fondo_path)


# Pregunta 3
Usando la detección de movimiento anterior, calcule el histograma proyectado por columnas
(suma de pixeles por columna) y el histograma proyectado por filas (suma de pixeles por fila)
Con esta información, encierre en una caja de dimensiones adecuadas el blob de la detección de
movimiento.



# Pregunta 4
Utilizando la información de la posición de la caja en el cuadro actual y la de los cuadros
anteriores, estime la posición del objeto para el cuadro siguiente. Dibuje la caja estimada y
compárela con la obtenida usando el procedimiento descrito en 3. Comente.



# Pregunta 5
Utilice el detector por diferencia de cuadros implementado y aplíquelo sobre la secuencia de
imágenes “estacionamiento”. Muestre ejemplos de detección de movimiento y explique la forma
que adoptan los pixeles de primer plano. Obtenga los bounding boxes de cada blob. Identifique
los principales problemas de utilizar este método.
